In [ ]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb



In [ ]:
df_raw = pd.read_csv('/Users/hariz/Desktop/TMDB-movie-analysis/extract_movie_features.csv')


In [ ]:
df = df_raw.copy()
df.columns

In [ ]:
df_feature = df[['original_language', 'release_year', 'release_month', 'release_day_of_week', 'budget', 'revenue', 'runtime', 'collection_name', 'is_part_of_franchise', 'primary_genre', 'primary_production_company', 'primary_country', 'director_name', 'cast']].copy()

df_feature.sort_values(by='collection_name', ascending=False).head(4)


In [ ]:
df_feature['primary_genre'].nunique()


In [ ]:
df_feature.info()

In [ ]:
# Convert raw budget and revenue to millions while keeping them numeric
df_feature['budget_m'] = df_feature['budget'] / 1e6
df_feature['revenue_m'] = df_feature['revenue'] / 1e6
df_feature['roi_ratio'] = df_feature['revenue'] / df['budget']
df_feature['log_budget'] = np.log1p(df_feature['budget'])
df_feature['log_revenue'] = np.log1p(df_feature['revenue'])

df_feature.describe()

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

features_m1 = [
    'log_budget',
    'runtime',
    'release_year',
    'release_month',
    'release_day_of_week',
    'original_language',
    'primary_genre',
    'primary_country',
    'primary_production_company',
    'is_part_of_franchise'
]

categorical_cols_m1 = [
    'original_language',
    'primary_genre',
    'primary_country',
    'primary_production_company'
]

numeric_cols_m1 = [c for c in features_m1 if c not in categorical_cols_m1]

In [19]:
# Prepare X and Targets
X = df[features_m1].copy()

# Target in Log Space
y_log = df['log_revenue'] if 'log_revenue' in df.columns else np.log1p(df['revenue'])
y_dollar = df['revenue']

# Handle missing values
X[numeric_cols_m1] = X[numeric_cols_m1].fillna(0)
X[categorical_cols_m1] = X[categorical_cols_m1].fillna('Unknown').astype(str)

# Train/Test Split (Fixed random_state=42 for comparability)
X_train, X_test, y_train_log, y_test_log, y_train_dollar, y_test_dollar = train_test_split(
    X, y_log, y_dollar, test_size=0.2, random_state=42
)

# Ordinal Encode Categoricals
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[categorical_cols_m1] = encoder.fit_transform(X_train[categorical_cols_m1])
X_test[categorical_cols_m1] = encoder.transform(X_test[categorical_cols_m1])

# Train Random Forest
print("Training Model 1 Baseline...")
model_1 = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
model_1.fit(X_train, y_train_log)

# Predict & Inverse Transform to Dollar Space
log_preds = model_1.predict(X_test)
dollar_preds = np.clip(np.expm1(log_preds), a_min=0, a_max=None)

# Calculate Dollar Metrics
rmse = np.sqrt(mean_squared_error(y_test_dollar, dollar_preds))
mae = mean_absolute_error(y_test_dollar, dollar_preds)
r2 = r2_score(y_test_dollar, dollar_preds)

print("\n--- Model 1 Baseline Performance ---")
print(f"Dollar RMSE: ${rmse:,.2f}")
print(f"Dollar MAE:  ${mae:,.2f}")
print(f"Dollar R²:   {r2:.4f}")

# Feature Importances
importance_m1 = pd.DataFrame({
    'Feature': features_m1,
    'Importance': model_1.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n--- Model 1 Feature Importances ---")
print(importance_m1)

Training Model 1 Baseline...

--- Model 1 Baseline Performance ---
Dollar RMSE: $105,193,172.17
Dollar MAE:  $50,385,471.90
Dollar R²:   0.5939

--- Model 1 Feature Importances ---
                      Feature  Importance
0                  log_budget    0.607390
1                release_year    0.073273
2        is_part_of_franchise    0.065289
3                     runtime    0.062859
4  primary_production_company    0.057141
5             primary_country    0.036601
6               release_month    0.035307
7         release_day_of_week    0.023058
8               primary_genre    0.022550
9           original_language    0.016532


In [ ]:
# ---------------------------------------------------------
# 1. Fixed Cast Count Parser
# ---------------------------------------------------------
def get_cast_count(val):
    if pd.isna(val) or val == '' or val is None:
        return 0
    if isinstance(val, str):
        val = val.strip()
        # Case A: If cast is stringified JSON/list of dicts like "[{'name': '...'}, ...]"
        if val.startswith('[') and val.endswith(']'):
            try:
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return len(parsed)
            except (ValueError, SyntaxError):
                pass
            # Fallback regex: Count occurrences of 'name' or 'character' keys in JSON
            names = re.findall(r"'name':\s*'([^']+)'", val)
            if names:
                return len(names)
        
        # Case B: Plain comma-separated list of actor names
        return len([x for x in val.split(',') if x.strip()])
    return 1

# ---------------------------------------------------------
# 2. Apply Clean Feature Engineering Directly to `df`
# ---------------------------------------------------------

# Corrected Cast Count
df['cast_count'] = df['cast'].apply(get_cast_count)

# Production Company Count
df['production_company_count'] = df['primary_production_company'].apply(
    lambda x: 0 if pd.isna(x) or str(x).strip() == '' else len(str(x).split(','))
)

# Genre Count (Check if you have a multi-genre column, otherwise count from primary)
if 'genres' in df.columns:
    df['genre_count'] = df['genres'].apply(
        lambda x: len(str(x).split(',')) if pd.notna(x) else 1
    )
else:
    df['genre_count'] = 1  # Standard fallback if only primary_genre exists

# Season
def month_to_season(month):
    if pd.isna(month): return 'Unknown'
    m = int(month)
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    elif m in [9, 10, 11]: return 'Fall'
    return 'Unknown'

df['season'] = df['release_month'].apply(month_to_season)

# Franchise Movie Count
if 'collection_name' in df.columns:
    franchise_counts = df['collection_name'].value_counts().to_dict()
    df['franchise_movie_count'] = df['collection_name'].map(franchise_counts).fillna(0).astype(int)
else:
    df['franchise_movie_count'] = df['is_part_of_franchise']

# Log Budget
df['log_budget'] = np.log1p(df['budget'])

# Verify again
print(df[['cast_count', 'production_company_count', 'genre_count', 'franchise_movie_count', 'season', 'log_budget']].head())

In [ ]:
def evaluate_model(df, feature_list, categorical_cols, model_name="Model"):
    """
    Trains a Random Forest model on log_revenue and evaluates in dollar metrics.
    """
    X = df[feature_list].copy()
    numeric_cols = [c for c in feature_list if c not in categorical_cols]
    
    y_log = df['log_revenue'] if 'log_revenue' in df.columns else np.log1p(df['revenue'])
    y_dollar = df['revenue']

    # Preprocess missing values
    X[numeric_cols] = X[numeric_cols].fillna(0)
    X[categorical_cols] = X[categorical_cols].fillna('Unknown').astype(str)
    
    # Train/Test Split (Fixed random_state=42 for direct baseline comparison)
    X_train, X_test, y_train_log, y_test_log, y_train_dollar, y_test_dollar = train_test_split(
        X, y_log, y_dollar, test_size=0.2, random_state=42
    )
    
    # Ordinal Encode Categoricals
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
    X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])
    
    # Train
    model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train_log)
    
    # Predict and Inverse Transform back to Dollars
    log_preds = model.predict(X_test)
    dollar_preds = np.clip(np.expm1(log_preds), a_min=0, a_max=None)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test_dollar, dollar_preds))
    mae = mean_absolute_error(y_test_dollar, dollar_preds)
    r2 = r2_score(y_test_dollar, dollar_preds)
    
    print(f"\n================ {model_name} ================")
    print(f"Dollar RMSE: ${rmse:,.2f}")
    print(f"Dollar MAE:  ${mae:,.2f}")
    print(f"Dollar R²:   {r2:.4f}")
    
    return {'model_name': model_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

In [ ]:
# Feature Sets
features_m1 = [
    'budget', 'runtime', 'release_year', 'release_month', 
    'primary_genre', 'original_language', 'primary_country', 'is_part_of_franchise'
]
cats_m1 = ['primary_genre', 'original_language', 'primary_country']

features_m2 = features_m1 + [
    'log_budget', 'season', 'genre_count', 
    'cast_count', 'production_company_count', 'franchise_movie_count'
]
cats_m2 = cats_m1 + ['season']

# Run Models
results_m1 = evaluate_model(df, features_m1, cats_m1, model_name="Model 1 (Baseline)")
results_m2 = evaluate_model(df, features_m2, cats_m2, model_name="Model 2 (Feature Engineering)")

# Print Comparison Table
comparison_df = pd.DataFrame([results_m1, results_m2]).set_index('model_name')
print("\n--- Side-by-Side Comparison ---")
print(comparison_df)